In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from conus_biomass import dir_info
from conus_biomass.make_figures import maps

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

In [ ]:
inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")

# Panel a: live biomass changes

In [ ]:
dir_data = dir_info.dir_model_output[:-1] + "_processed/"

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt
western_states = maps.SHP_WESTERN.to_crs(crs)

In [ ]:
def get_filtered_output(
    year,
    fdir=dir_data,
    ftype="nc",
    vartype="predicted_biomass_filtered_",
):
    fname = fdir + vartype + str(year)
    if ftype == "zarr":
        ds = xr.open_zarr(fname + ".zarr")
    elif ftype == "nc":
        ds = xr.load_dataset(fname + "_0000.nc")
    return ds


ds_end = get_filtered_output(year=2022)
ds_start = get_filtered_output(year=2005)
delta_biomass = ds_end["predicted_biomass"] - ds_start["predicted_biomass"]

In [ ]:
inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")
inputs = inputs.rio.write_crs(crs)

In [ ]:
burned = inputs["years_after_fire"].sel(year=2022) <= 17

In [ ]:
burned_clipped = burned.rio.clip(western_states.geometry)

In [ ]:
biomass_start_clipped = (
    ds_start["predicted_biomass"].rio.write_crs(crs).rio.clip(western_states.geometry)
)

In [ ]:
delta_biomass = delta_biomass.rio.write_crs(crs)

In [ ]:
delta_biomass_clipped = delta_biomass.rio.clip(western_states.geometry)

In [ ]:
maps.plot_map(
    delta_biomass_clipped,  # .mean(),
    shp=western_states,
    latlon=False,
    title="Change in Live Aboveground Biomass from 2005-2020",
    cbar_label=r"$\Delta$ Live AGB (Mg/ha)",
    cmap=plt.cm.BrBG,
    clims=[-60, 60],
    savefig=None,
)
# plt.savefig('Figure2.pdf')

In [ ]:
delta_biomass_burned = (
    delta_biomass_clipped.where(burned_clipped).where(biomass_start_clipped > 1)
    / biomass_start_clipped
)
delta_biomass_unburned = (
    delta_biomass_clipped.where(~burned_clipped).where(biomass_start_clipped > 1)
    / biomass_start_clipped
)

In [ ]:
num_burned = burned_clipped.where(~np.isnan(biomass_start_clipped)).sum()

In [ ]:
num_unburned = (~burned_clipped).where(~np.isnan(biomass_start_clipped)).sum()

In [ ]:
print(num_unburned / (num_burned + num_unburned))
print(num_burned / (num_burned + num_unburned))

In [ ]:
plt.hist(
    delta_biomass_burned.values.flatten() * 100,
    bins=np.arange(-100, 250, 2),
    alpha=0.3,
    density=False,
    color="saddlebrown",
    label="Burned",
)
plt.hist(
    delta_biomass_unburned.values.flatten() * 100,
    bins=np.arange(-100, 250, 2),
    alpha=0.3,
    density=False,
    color="teal",
    label="Unburned",
)  # )#, bins=np.arange(-202.5, 100, 5), alpha=0.3, density=True)
plt.axvline(x=0, linestyle=":", color="k", zorder=0)
plt.xlabel("Percent change from 2005-2022")
plt.legend(framealpha=0)
plt.ylabel("Number of grid cells")

In [ ]:
import seaborn as sns

sns.displot(
    delta_biomass_burned.values.flatten()
)  # , bins=np.arange(-202.5, 100, 5), alpha=0.3, density=True)
sns.displot(
    delta_biomass_unburned.values.flatten()
)  # , bins=np.arange(-202.5, 100, 5), alpha=0.3, density=True)

# Exploratory: fractional change

In [ ]:
maps.plot_map(
    delta_biomass.coarsen(x=5, y=5, boundary="pad").mean()
    * 100
    / ds_start["predicted_biomass"].coarsen(x=5, y=5, boundary="pad").mean(),
    shp=maps.SHP,
    latlon=False,
    title="Percent Change in Live Aboveground Biomass from 2005-2022",
    cbar_label=r"$\Delta$ Live AGB (Mg/ha)",
    cmap=plt.cm.BrBG,
    clims=[-100, 100],
    savefig=None,
)
# plt.savefig('Figure2.pdf')

In [ ]:
biomass_delta_pct = delta_biomass * 100 / ds_start["predicted_biomass"]

# Panel b: years after disturbance

In [ ]:
years_post_disturbance = xr.open_zarr(
    dir_info.dir_processed
    + "data_on_ref_grid/100m/years_post_disturbance/"
    + "years_post_disturbance_2022.zarr"
)["years_post_disturbance"]

In [ ]:
def get_2d_forest_filter(year: int):
    forest_filter = xr.open_dataset(
        dir_info.dir_processed
        + "data_on_ref_grid/"
        + "/100m/forest_remaining_forest/"
        + "forest_remaining_forest"
        + str(year)
        + "_average.nc"
    )["__xarray_dataarray_variable__"][0, :, :]
    return forest_filter

In [ ]:
forest_filter = get_2d_forest_filter(year=2005)

In [ ]:
maps.plot_map(
    years_post_disturbance.where(forest_filter > 1).coarsen(x=100, y=100, boundary="pad").mean(),
    shp=maps.SHP,
    latlon=False,
    title="",
    cbar_label="Years since fire",
    cmap=plt.cm.copper_r,
    clims=[0, 60],
    savefig=None,
)
# plt.savefig('Years_since_fire.pdf')

# Show tiles

In [ ]:
plt.figure(figsize=(15, 9))
(delta_biomass).coarsen(x=100, y=100, boundary="pad").mean().plot(vmax=55, cmap=plt.cm.BrBG)
for ybound in np.arange(np.nanmax(delta_biomass.y), np.nanmin(delta_biomass.y), -2000 * 100):
    plt.axhline(y=ybound, color="k", linestyle="--", alpha=0.3, linewidth=1)
for xbound in np.arange(np.nanmin(delta_biomass.x), np.nanmax(delta_biomass.x), 2000 * 100):
    plt.axvline(x=xbound, color="k", linestyle="--", alpha=0.3, linewidth=1)